# Position and order sensitivity of a small model: full pipeline (Colab, one-click)

**Status: written but NOT yet run in Colab.** The offline logic was tested with dummy data and a dummy model; the model, GPU and download steps are untested. Expect the first run to need fixes: paste any error back.

**How to use**
1. Runtime > Change runtime type > **GPU** (a T4 is enough for the model; availability is not guaranteed).
2. In the settings cell, `MODE = "reduced"` is the shorter real run (3 gold positions, original and one shuffled distractor order, 300 evaluation questions, about 2,950 prompts). `"quick"` is a tiny trial only for finding bugs (its results are not for the paper); `"full"` is the complete design (about 6,100 prompts). The evaluation questions are the same in every mode.
3. Choose **Run all** and allow Google Drive access when asked. The run is resumable: if Colab disconnects, run all again and it continues from `raw_outputs.jsonl` on Drive. Each mode writes to its own folder, so modes never mix.

**What it does:** clones the original repository at a fixed commit, loads the 20-document files, validates them, selects eligible questions, makes a seeded dev/eval split, builds prompts with the original functions, runs the model (greedy), scores with the original metric, stops if a sanity gate fails, audits the predictions table, then writes `results.csv`, figures and an environment record to Drive.

**What it does not do:** interpret results. Read the numbers with the evidence rules in the protocol.

In [ ]:
# ===== SETTINGS (edit only this cell) =====
import os, sys, json, subprocess, time, hashlib

MODE = "reduced"      # "quick" = tiny trial of the pipeline | "reduced" = the shorter real run | "full" = the complete design
RETRIEVAL_FILE_ON_DRIVE = ""   # optional: path on Drive of an uploaded nq-open-contriever-msmarco-retrieved-documents.jsonl.gz
GOLD0_DRIVE_LINK = "PASTE_YOUR_DRIVE_SHARE_LINK_FOR_nq-open-20_total_documents_gold_at_0.jsonl.gz"
                               # share link of the ONE file nq-open-20_total_documents_gold_at_0.jsonl.gz (downloaded automatically)
GOLD0_EXPECTED_SHA256 = "69a8dd87a45f07fb3ed9b804917a89db975cd6e488682fc838edca029cc3881f"   # hash of the copy inspected while building this notebook
GOLD0_FILE_ON_DRIVE = ""       # alternative to the link, a path on mounted Drive. EASIEST: path on Drive of ONE file, nq-open-20_total_documents_gold_at_0.jsonl.gz. The other four
                               # position files are derived from it (same distractors, gold passage moved). Leave "" to use the repository.
QA_DIR_ON_DRIVE = ""           # optional: Drive folder that already holds all five position files
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
REPO_URL = "https://github.com/nelson-liu/lost-in-the-middle"
REPO_COMMIT = "29b8a6d042ce29abccee3db1a73171a107d7e6af"   # commit printed by the author's own clone
ALL_POSITIONS = [0, 4, 9, 14, 19]   # the original files exist for these gold positions
TOTAL_DOCS = 20
SPLIT_SEED, BOOT_SEED = 1234, 777
MAX_NEW_TOKENS = 100  # default of the original response scripts
_DESIGN = {"quick":   dict(N_DEV=20,  N_EVAL=30,  N_RERUN=10,  POSITIONS=[0, 4, 9, 14, 19], ORDERINGS=["O1", "O2"]),
           "reduced": dict(N_DEV=50,  N_EVAL=300, N_RERUN=100, POSITIONS=[0, 9, 19],         ORDERINGS=["O1"]),
           "full":    dict(N_DEV=100, N_EVAL=300, N_RERUN=100, POSITIONS=[0, 4, 9, 14, 19], ORDERINGS=["O1", "O2"])}
assert MODE in _DESIGN, "MODE must be quick, reduced or full"
N_DEV, N_EVAL, N_RERUN = _DESIGN[MODE]["N_DEV"], _DESIGN[MODE]["N_EVAL"], _DESIGN[MODE]["N_RERUN"]
POSITIONS, ORDERINGS = _DESIGN[MODE]["POSITIONS"], _DESIGN[MODE]["ORDERINGS"]
TAG = MODE
DRIVE_DIR = "/content/drive/MyDrive/position_order_run"
WORK = "/content/work"
OUT = f"{DRIVE_DIR}/{TAG}"
N_PROMPTS = N_DEV * 5 + N_EVAL * (2 + len(POSITIONS) * (1 + len(ORDERINGS))) + min(N_RERUN, N_EVAL) * len(POSITIONS)
print("MODE =", MODE, "| gold positions", POSITIONS, "| orderings O0 +", ORDERINGS, "| dev/eval/rerun questions:", N_DEV, N_EVAL, N_RERUN)
print("prompts to run in this mode:", N_PROMPTS)

## 1. Drive, GPU and packages

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)
os.makedirs(f"{WORK}/position-order-study/src/positionorder", exist_ok=True)
import torch
assert torch.cuda.is_available(), "No GPU found. Use Runtime > Change runtime type > GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0), "| compute capability", torch.cuda.get_device_capability(0))
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xopen", "regex", "accelerate"],
                   capture_output=True, text=True)
print(r.stdout[-400:], r.stderr[-400:])
import transformers, pydantic
print("torch", torch.__version__, "| transformers", transformers.__version__, "| pydantic", pydantic.__version__)

## 2. Project code (written to the Colab disk; read it before trusting it)

In [ ]:
%%writefile /content/work/position-order-study/src/positionorder/__init__.py
"""Position and order sensitivity study: support code."""

__all__ = ["seeds"]

In [ ]:
%%writefile /content/work/position-order-study/src/positionorder/seeds.py
"""Seed controls for the position/order study.

Design rules
------------
* Never use Python's built-in hash() for seeds: it is salted per process.
* Derive every random draw from (base seed, stable identifiers) with SHA-256,
  so results do not depend on iteration order, resumption, or process restarts.
* Seeds are set in configs/protocol_v0.1.yaml before any result exists and are
  not tuned afterwards. Changing one is a logged protocol amendment.
"""

from __future__ import annotations

import hashlib
import random
from typing import List

import numpy as np


def derive_seed(base_seed: int, *parts: object) -> int:
    """Return a 64-bit seed derived from a base seed and stable identifiers."""
    key = ":".join([str(base_seed)] + [str(p) for p in parts]).encode("utf-8")
    return int.from_bytes(hashlib.sha256(key).digest()[:8], "big")


def set_global_seed(seed: int) -> None:
    """Seed Python's and NumPy's global generators.

    Note: PYTHONHASHSEED only takes effect if set before the interpreter starts,
    which is why no result in this project depends on hash().
    """
    random.seed(seed)
    np.random.seed(seed % (2**32))


def distractor_permutation(ordering_seed: int, question_id: str, n: int = 19) -> List[int]:
    """Fixed permutation of n distractors for one question and one ordering seed.

    The same (ordering_seed, question_id) always gives the same permutation,
    whatever else has been generated before. The permutation is NOT a function of
    the gold position, so only the gold index moves between conditions.
    """
    rng = np.random.Generator(np.random.PCG64(derive_seed(ordering_seed, question_id)))
    return [int(i) for i in rng.permutation(n)]

In [ ]:
%%writefile /content/work/position-order-study/src/positionorder/data.py
"""Data loading and validation (cycle 1).

Reads the multi-document QA files of the original Lost-in-the-Middle protocol
(one gzipped JSON-lines file per gold position) and checks them against the
protocol's assumptions. It excludes nothing: it only reports.

File format (from the original repository's README, as read in an earlier stage):
    one JSON object per line with keys
        question (str), answers (list[str]), ctxs (list of passages),
        nq_annotated_gold (dict, not used here)
    each passage: id, title, text, score, hasanswer, original_retrieval_index, isgold

Shapes
------
    load_qa_file(path)         -> list[QAExample]          length N (questions in file)
    QAExample.passages         -> list[Passage]             length 20 in the main design
    validate_file(...)         -> dict (JSON-serialisable)
    check_across_positions(..) -> dict (JSON-serialisable)

This module uses only the standard library.
"""

from __future__ import annotations

import gzip
import hashlib
import json
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional

PROBLEM_CODES = (
    "bad_question",
    "no_answers",
    "ctx_count",
    "gold_count",
    "gold_index_mismatch",
    "distractor_has_answer",
    "empty_text",
    "duplicate_text",
)


class DataFormatError(ValueError):
    """The file is not in the expected structure (cannot be analysed)."""


@dataclass(frozen=True)
class Passage:
    title: str
    text: str
    isgold: bool
    hasanswer: bool
    original_retrieval_index: Optional[int] = None
    score: Optional[float] = None
    id: Optional[str] = None


@dataclass(frozen=True)
class QAExample:
    question: str
    answers: List[str]
    passages: List[Passage]

    @property
    def qid(self) -> str:
        return question_id(self.question)

    def gold_positions(self) -> List[int]:
        return [i for i, p in enumerate(self.passages) if p.isgold]


# --------------------------------------------------------------------------- ids


def normalize_question(question: str) -> str:
    """Lowercase, Unicode-normalise, drop punctuation, collapse whitespace."""
    q = unicodedata.normalize("NFKC", question).lower()
    q = re.sub(r"[^\w\s]", "", q)
    return re.sub(r"\s+", " ", q).strip()


def question_id(question: str) -> str:
    """Stable 16-hex-character id from the normalised question text.

    The data files carry no question id, so this one is derived. It is used for
    splits and for per-question seeds, and it does not depend on file order.
    """
    return hashlib.sha256(normalize_question(question).encode("utf-8")).hexdigest()[:16]


# ----------------------------------------------------------------------- loading


def _open_text(path: Path):
    if str(path).endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "r", encoding="utf-8")


def _parse_example(raw: dict, where: str) -> QAExample:
    for key in ("question", "answers", "ctxs"):
        if key not in raw:
            raise DataFormatError(f"{where}: missing key '{key}'")
    if not isinstance(raw["ctxs"], list):
        raise DataFormatError(f"{where}: 'ctxs' is not a list")
    passages: List[Passage] = []
    for j, c in enumerate(raw["ctxs"]):
        for key in ("text", "isgold", "hasanswer"):
            if key not in c:
                raise DataFormatError(f"{where}: passage {j} missing key '{key}'")
        for key in ("isgold", "hasanswer"):
            if not isinstance(c[key], bool):
                raise DataFormatError(
                    f"{where}: passage {j} field '{key}' is {type(c[key]).__name__} "
                    f"({c[key]!r}), expected a JSON boolean"
                )
        passages.append(
            Passage(
                title=str(c.get("title", "")),
                text=str(c["text"]),
                isgold=c["isgold"],
                hasanswer=c["hasanswer"],
                original_retrieval_index=c.get("original_retrieval_index"),
                score=c.get("score"),
                id=None if c.get("id") is None else str(c["id"]),
            )
        )
    return QAExample(
        question=str(raw["question"]),
        answers=[str(a) for a in raw["answers"]],
        passages=passages,
    )


def load_qa_file(path: Path | str, limit: Optional[int] = None) -> List[QAExample]:
    """Read one file. Raises DataFormatError on structural problems."""
    path = Path(path)
    out: List[QAExample] = []
    with _open_text(path) as fh:
        for line_no, line in enumerate(fh, start=1):
            line = line.strip()
            if not line:
                continue
            where = f"{path.name} line {line_no}"
            try:
                raw = json.loads(line)
            except json.JSONDecodeError as exc:
                raise DataFormatError(f"{where}: invalid JSON ({exc.msg})") from exc
            out.append(_parse_example(raw, where))
            if limit is not None and len(out) >= limit:
                break
    return out


# -------------------------------------------------------------------- validation


def validate_example(
    ex: QAExample, total_documents: int = 20, expected_gold_index: Optional[int] = None
) -> List[str]:
    """Return the list of problem codes found in one example (empty list = clean)."""
    problems: List[str] = []
    if not normalize_question(ex.question):
        problems.append("bad_question")
    if not [a for a in ex.answers if a.strip()]:
        problems.append("no_answers")
    if len(ex.passages) != total_documents:
        problems.append("ctx_count")
    golds = ex.gold_positions()
    if len(golds) != 1:
        problems.append("gold_count")
    elif expected_gold_index is not None and golds[0] != expected_gold_index:
        problems.append("gold_index_mismatch")
    if any(p.hasanswer for p in ex.passages if not p.isgold):
        problems.append("distractor_has_answer")
    if any(not p.text.strip() for p in ex.passages):
        problems.append("empty_text")
    texts = [p.text.strip() for p in ex.passages]
    if len(set(texts)) != len(texts):
        problems.append("duplicate_text")
    return problems


def validate_file(
    path: Path | str,
    total_documents: int = 20,
    expected_gold_index: Optional[int] = None,
    examples: Optional[List[QAExample]] = None,
) -> dict:
    """Validate every example in a file. Nothing is excluded or modified.

    Pass `examples` (already loaded from `path`) to avoid reading the file twice.
    """
    if examples is None:
        examples = load_qa_file(path)
    counts = {code: 0 for code in PROBLEM_CODES}
    sample_ids: Dict[str, List[str]] = {code: [] for code in PROBLEM_CODES}
    clean = 0
    for ex in examples:
        found = validate_example(ex, total_documents, expected_gold_index)
        if not found:
            clean += 1
        for code in found:
            counts[code] += 1
            if len(sample_ids[code]) < 5:
                sample_ids[code].append(ex.qid)
    qids = [ex.qid for ex in examples]
    return {
        "file": str(path),
        "expected_gold_index": expected_gold_index,
        "n_examples": len(examples),
        "n_unique_questions": len(set(qids)),
        "n_duplicate_question_rows": len(qids) - len(set(qids)),
        "n_clean": clean,
        "problem_counts": counts,
        "problem_example_ids": sample_ids,
    }


def check_across_positions(by_position: Dict[int, List[QAExample]]) -> dict:
    """Check that files for different gold positions describe the same items.

    The protocol assumes that, for a given question, the 19 distractors and the
    gold passage are the same in every file and only the gold index moves. This
    function tests that assumption; it does not assume it.
    """
    positions = sorted(by_position)
    maps: Dict[int, Dict[str, QAExample]] = {
        p: {ex.qid: ex for ex in by_position[p]} for p in positions
    }
    all_ids = set().union(*[set(m) for m in maps.values()]) if maps else set()
    in_all = set.intersection(*[set(m) for m in maps.values()]) if maps else set()

    gold_differs: List[str] = []
    set_differs: List[str] = []
    seq_differs: List[str] = []
    for qid in sorted(in_all):
        exs = [maps[p][qid] for p in positions]
        gold_texts = {
            tuple(q.text.strip() for q in ex.passages if q.isgold) for ex in exs
        }
        dist_seqs = [
            tuple(q.text.strip() for q in ex.passages if not q.isgold) for ex in exs
        ]
        if len(gold_texts) != 1:
            gold_differs.append(qid)
        if len({frozenset(s) for s in dist_seqs}) != 1:
            set_differs.append(qid)
        if len(set(dist_seqs)) != 1:
            seq_differs.append(qid)
    return {
        "positions": positions,
        "n_questions_union": len(all_ids),
        "n_questions_in_all_files": len(in_all),
        "n_questions_missing_from_some_file": len(all_ids) - len(in_all),
        "n_gold_text_differs": len(gold_differs),
        "n_distractor_set_differs": len(set_differs),
        "n_distractor_sequence_differs": len(seq_differs),
        "example_ids_gold_differs": gold_differs[:5],
        "example_ids_distractor_set_differs": set_differs[:5],
        "example_ids_distractor_sequence_differs": seq_differs[:5],
    }

In [ ]:
%%writefile /content/work/position-order-study/src/positionorder/scoring.py
"""Scorer that mirrors the original repository's metric (cycle 6).

Source of the logic: `scripts/evaluate_qa_responses.py` and `lost_in_the_middle/metrics.py` of
the Lost-in-the-Middle repository (MIT licence), pasted by the project author. This file is a
re-implementation for offline testing. On Colab the repository function is called as well and
the two must agree on every output (see cross_check). The original is the reference.

Rules (VERIFIED from the pasted code):
  1. Only the text before the first newline of the reply is scored, after strip().
  2. Normalisation (SQuAD): lowercase, remove ASCII punctuation, remove the words a/an/the, fix spaces.
  3. Correct (1.0) if ANY normalised accepted answer is a SUBSTRING of the normalised reply, else 0.0.
Known properties of rule 3 (reported, not changed): matching is by character substring, not by word,
so a short answer such as "art" matches inside "start"; an accepted answer that normalises to the
empty string matches every reply.
"""
from __future__ import annotations

import re
import string
from typing import Iterable, List

_PUNC = set(string.punctuation)


def normalize_answer(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in s if ch not in _PUNC)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def first_line(model_answer: str) -> str:
    return model_answer.split("\n")[0].strip()


def best_subspan_em(prediction: str, ground_truths: List[str]) -> float:
    p = normalize_answer(prediction)
    for g in ground_truths:
        if normalize_answer(g).lower() in p.lower():
            return 1.0
    return 0.0


def score_reply(model_answer: str, answers: List[str]) -> int:
    """The 0/1 value written to the `correct` column."""
    return int(best_subspan_em(first_line(model_answer), answers))


def risky_answers(answers: Iterable[str], min_len: int = 4) -> List[str]:
    """Accepted answers that are empty or very short after normalisation (substring-leniency risk)."""
    return [a for a in answers if len(normalize_answer(a)) < min_len]


def cross_check(rows, reference_fn) -> List[int]:
    """rows: iterable of (model_answer, answers). reference_fn: the repository's best_subspan_em.
    Returns indices where this scorer and the original disagree (must be empty)."""
    bad = []
    for i, (reply, answers) in enumerate(rows):
        mine = score_reply(reply, answers)
        ref = int(reference_fn(prediction=first_line(reply), ground_truths=answers))
        if mine != ref:
            bad.append(i)
    return bad

In [ ]:
%%writefile /content/work/position-order-study/src/positionorder/prompts_local.py
"""Local copy of the original repository's QA prompt construction (cycle 6).

Source: `lost_in_the_middle/prompting.py` and `prompts/qa.prompt`, `prompts/closedbook_qa.prompt`
(MIT licence), printed by the project author. Used (a) offline for tests and (b) as a cross-check
against the repository's own functions in the notebook. When the repository functions can be
imported, they are the reference and the two must give identical strings.
Only the plain QA prompt and the closed-book prompt are copied; the other templates are not used.
"""
from __future__ import annotations

import dataclasses
from dataclasses import dataclass
from typing import List, Optional

# Template files are read with trailing newlines removed (rstrip("\n")) in the original.
QA_TEMPLATE = (
    "Write a high-quality answer for the given question using only the provided search results "
    "(some of which might be irrelevant).\n\n{search_results}\n\nQuestion: {question}\nAnswer:"
)
CLOSEDBOOK_TEMPLATE = "Question: {question}\nAnswer:"


@dataclass(frozen=True)
class Document:
    title: str
    text: str
    id: Optional[str] = None
    score: Optional[float] = None
    hasanswer: Optional[bool] = None
    isgold: Optional[bool] = None
    original_retrieval_index: Optional[int] = None

    @classmethod
    def from_dict(cls, data: dict) -> "Document":
        names = {f.name for f in dataclasses.fields(cls)}
        return cls(**{k: v for k, v in data.items() if k in names})


def get_qa_prompt(question: str, documents: List[Document], mention_random_ordering: bool = False,
                  query_aware_contextualization: bool = False) -> str:
    if not question:
        raise ValueError("question must be truthy")
    if not documents:
        raise ValueError("documents must be truthy")
    if mention_random_ordering or query_aware_contextualization:
        raise NotImplementedError("only the plain qa.prompt is copied locally")
    formatted = [f"Document [{i + 1}](Title: {d.title}) {d.text}" for i, d in enumerate(documents)]
    return QA_TEMPLATE.format(question=question, search_results="\n".join(formatted))


def get_closedbook_qa_prompt(question: str) -> str:
    if not question:
        raise ValueError("question must be truthy")
    return CLOSEDBOOK_TEMPLATE.format(question=question)

In [ ]:
%%writefile /content/work/position-order-study/src/positionorder/evaluate.py
"""Evaluation: audit first, then metrics (cycle 5).

Input is a long-format predictions table, one row per (question, condition, gold position, run).
Required columns (see REQUIRED_COLUMNS). `correct` must come from the frozen scorer; this module
does not score text. Nothing is interpreted here: it audits, computes, and exports.

Metric directions (REGISTRY): accuracy higher is better; flip_rate lower means more stable.
Uncertainty: percentile bootstrap over questions (pairs kept together). Paired comparisons also
get an exact McNemar test. p-values are reported unadjusted and Holm-adjusted; no result is
labelled "significant" by this code, and the paired tests assume questions are independent.
"""
from __future__ import annotations

import hashlib
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
from scipy.stats import binomtest

REQUIRED_COLUMNS = [
    "qid", "split", "condition", "gold_index", "run_id", "correct",
    "model_id", "template_id", "scorer_version", "decoding", "max_new_tokens",
    "ctx_size", "n_prompt_tokens", "truncated",
]
# Settings that must be identical in every row of a primary comparison.
COMPARATOR_COLUMNS = ["model_id", "template_id", "scorer_version", "decoding",
                      "max_new_tokens", "ctx_size"]
REGISTRY = {"accuracy": "higher", "flip_rate": "lower"}
CONTROLS = ("C1", "C2")  # closed-book, oracle: no gold_index; may use their own template_id
SUBSET_CONDITIONS = ("R",)  # rerun: may cover only a subset of the evaluation questions


@dataclass
class Finding:
    level: str   # "ERROR" blocks metrics; "WARN" is reported
    code: str
    message: str


@dataclass
class AuditReport:
    findings: List[Finding] = field(default_factory=list)

    def add(self, level, code, message):
        self.findings.append(Finding(level, code, message))

    @property
    def ok(self) -> bool:
        return not any(f.level == "ERROR" for f in self.findings)


def sha256_file(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def audit(df: pd.DataFrame, positions=(0, 4, 9, 14, 19), eval_split="eval",
          primary_conditions=None) -> AuditReport:
    """Run every check before any metric. Returns findings; ERROR means stop."""
    rep = AuditReport()
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        rep.add("ERROR", "columns", f"missing columns: {missing_cols}")
        return rep
    # 1. missing values (gold_index may be empty only for controls)
    for c in REQUIRED_COLUMNS:
        if c == "gold_index":
            bad = df[df[c].isna() & ~df["condition"].isin(CONTROLS)]
        else:
            bad = df[df[c].isna()]
        if len(bad):
            rep.add("ERROR", "missing_values", f"{len(bad)} rows with missing '{c}'")
    # 2. label domain: correct is 0/1, truncated is boolean-like
    if not set(df["correct"].dropna().unique()) <= {0, 1}:
        rep.add("ERROR", "label_domain", "'correct' has values other than 0/1")
    if not set(df["truncated"].dropna().astype(int).unique()) <= {0, 1}:
        rep.add("ERROR", "label_domain", "'truncated' is not 0/1")
    elif int(df["truncated"].astype(int).sum()) > 0:
        rep.add("ERROR", "truncation", f"{int(df['truncated'].astype(int).sum())} rows are truncated prompts")
    if rep.findings and not rep.ok:
        return rep  # later checks assume clean basics
    # 3. partitions and leakage
    splits = set(df["split"].unique())
    if not splits <= {"dev", "eval", "replication"}:
        rep.add("ERROR", "split_values", f"unknown split labels: {sorted(splits - {'dev','eval','replication'})}")
    per_q = df.groupby("qid")["split"].nunique()
    leaked = per_q[per_q > 1]
    if len(leaked):
        rep.add("ERROR", "leakage", f"{len(leaked)} question ids appear in more than one split, e.g. {list(leaked.index[:3])}")
    # 4. duplicates
    key = ["qid", "condition", "gold_index", "run_id"]
    dups = df.fillna({"gold_index": -1}).duplicated(key, keep=False)
    if dups.any():
        rep.add("ERROR", "duplicates", f"{int(dups.sum())} rows share (qid, condition, gold_index, run_id)")
    # 5. gold index domain
    non_ctrl = df[~df["condition"].isin(CONTROLS)]
    badpos = non_ctrl[~non_ctrl["gold_index"].isin(positions)]
    if len(badpos):
        rep.add("ERROR", "gold_index", f"{len(badpos)} rows with gold_index outside {list(positions)}")
    if rep.ok is False:
        return rep
    ev = df[df["split"] == eval_split]
    if ev.empty:
        rep.add("ERROR", "no_eval_rows", f"no rows with split == '{eval_split}'")
        return rep
    # 6. completeness: every eval question has every cell of every non-control condition
    cells = ev.fillna({"gold_index": -1}).groupby(["condition", "gold_index", "run_id"])["qid"].apply(set)
    ref_q = set(ev["qid"])
    for cell, qs in cells.items():
        if cell[0] in SUBSET_CONDITIONS:
            if not qs <= ref_q:
                rep.add("ERROR", "incomplete_cell", f"cell {cell}: {len(qs - ref_q)} questions outside the evaluation set")
            continue
        if qs != ref_q:
            rep.add("ERROR", "incomplete_cell",
                    f"cell {cell}: {len(ref_q - qs)} eval questions missing, {len(qs - ref_q)} extra")
    # 7. incompatible comparator settings
    primary = ev[~ev["condition"].isin(CONTROLS)]
    if primary_conditions is not None:
        primary = primary[primary["condition"].isin(primary_conditions)]
    for c in COMPARATOR_COLUMNS:
        vals = primary[c].astype(str).unique()
        if len(vals) > 1:
            rep.add("ERROR", "comparator_mismatch",
                    f"'{c}' differs across primary conditions: {sorted(vals)}")
    for c in ("model_id", "scorer_version", "decoding", "max_new_tokens", "ctx_size"):
        cv = ev[ev["condition"].isin(CONTROLS)][c].astype(str).unique()
        pv = primary[c].astype(str).unique()
        if len(cv) and len(pv) and set(cv) != set(pv):
            rep.add("WARN", "control_setting", f"controls use different '{c}' ({sorted(cv)}) than primary ({sorted(pv)})")
    return rep


def _boot_idx(n: int, n_boot: int, seed: int) -> np.ndarray:
    rng = np.random.Generator(np.random.PCG64(seed))
    return rng.integers(0, n, size=(n_boot, n))


def cell_vectors(ev: pd.DataFrame, run_id=0) -> Dict[tuple, pd.Series]:
    """(condition, gold_index) -> correct vector indexed by qid, qids sorted."""
    ev = ev[ev["run_id"] == run_id].fillna({"gold_index": -1})
    out = {}
    for (cond, g), sub in ev.groupby(["condition", "gold_index"]):
        out[(cond, int(g))] = sub.set_index("qid")["correct"].astype(int).sort_index()
    return out


def accuracy_ci(x: np.ndarray, n_boot=10000, seed=777, alpha=0.05):
    idx = _boot_idx(len(x), n_boot, seed)
    means = x[idx].mean(axis=1)
    return x.mean(), *np.quantile(means, [alpha / 2, 1 - alpha / 2])


def paired_compare(a: np.ndarray, b: np.ndarray, n_boot=10000, seed=777, alpha=0.05) -> dict:
    """a, b: 0/1 vectors on the SAME questions in the SAME order. Reports a - b."""
    if a.shape != b.shape:
        raise ValueError("paired vectors must have equal length (same questions)")
    diff = a - b
    idx = _boot_idx(len(a), n_boot, seed)
    boots = diff[idx].mean(axis=1)
    n01 = int(((a == 0) & (b == 1)).sum())   # a wrong, b right
    n10 = int(((a == 1) & (b == 0)).sum())   # a right, b wrong
    p = 1.0 if n01 + n10 == 0 else binomtest(n10, n10 + n01, 0.5).pvalue
    return {"diff": float(diff.mean()), "ci_low": float(np.quantile(boots, alpha / 2)),
            "ci_high": float(np.quantile(boots, 1 - alpha / 2)),
            "a_right_b_wrong": n10, "a_wrong_b_right": n01, "p_exact_mcnemar": float(p),
            "flip_rate": (n01 + n10) / len(a)}


def holm(pvals: List[float]) -> List[float]:
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * pvals[i])
        adj[i] = min(1.0, running)
    return adj.tolist()


def compute_results(df: pd.DataFrame, eval_split="eval", n_boot=10000, seed=777,
                    input_sha256="", pairs=None) -> pd.DataFrame:
    """Build the long results table. `pairs` = list of ((condA,gA),(condB,gB)) to compare."""
    ev = df[df["split"] == eval_split]
    vec = cell_vectors(ev)
    rows = []
    base = {"split": eval_split, "bootstrap_seed": seed, "n_boot": n_boot, "input_sha256": input_sha256}
    for (cond, g), s in sorted(vec.items()):
        est, lo, hi = accuracy_ci(s.to_numpy(), n_boot, seed)
        rows.append({**base, "table": "accuracy", "metric": "accuracy", "direction": REGISTRY["accuracy"],
                     "condition": cond, "gold_index": None if g == -1 else g, "comparator": "",
                     "n": len(s), "estimate": est, "ci_low": lo, "ci_high": hi})
    pair_rows = []
    for (ka, kb) in (pairs or []):
        sa, sb = vec[ka], vec[kb]
        if ka[0] in SUBSET_CONDITIONS or kb[0] in SUBSET_CONDITIONS:
            common = sa.index.intersection(sb.index)      # rerun covers a subset: compare on shared questions
            if len(common) == 0:
                raise ValueError(f"no shared questions between {ka} and {kb}")
            sa, sb = sa.loc[common], sb.loc[common]
        elif not sa.index.equals(sb.index):
            raise ValueError(f"question ids differ between {ka} and {kb}")
        r = paired_compare(sa.to_numpy(), sb.to_numpy(), n_boot, seed)
        pair_rows.append((ka, kb, r, len(sa)))
    adj = holm([r["p_exact_mcnemar"] for _, _, r, _ in pair_rows]) if pair_rows else []
    for (ka, kb, r, n), pa in zip(pair_rows, adj):
        rows.append({**base, "table": "paired", "metric": "accuracy_difference", "direction": REGISTRY["accuracy"],
                     "condition": ka[0], "gold_index": None if ka[1] == -1 else ka[1],
                     "comparator": f"{kb[0]}@{None if kb[1] == -1 else kb[1]}", "n": n,
                     "estimate": r["diff"], "ci_low": r["ci_low"], "ci_high": r["ci_high"],
                     "a_right_b_wrong": r["a_right_b_wrong"], "a_wrong_b_right": r["a_wrong_b_right"],
                     "p_unadjusted": r["p_exact_mcnemar"], "p_holm": pa, "test": "exact McNemar",
                     "flip_rate": r["flip_rate"]})
    return pd.DataFrame(rows)


def plot_accuracy(results: pd.DataFrame, out_stem: Path, title="Accuracy by gold position", conditions=None):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    acc = results[(results["table"] == "accuracy") & results["gold_index"].notna()]
    acc = acc[acc["condition"].isin(conditions)] if conditions else acc[~acc["condition"].isin(SUBSET_CONDITIONS)]
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    markers = ["o", "s", "^", "D", "v"]
    groups = list(acc.groupby("condition"))
    for i, (cond, sub) in enumerate(groups):
        sub = sub.sort_values("gold_index")
        x = sub["gold_index"].to_numpy() + (i - (len(groups) - 1) / 2) * 0.35  # dodge to avoid overlap
        ax.errorbar(x, sub["estimate"], yerr=[sub["estimate"] - sub["ci_low"], sub["ci_high"] - sub["estimate"]],
                    marker=markers[i % 5], capsize=3, linewidth=1.2, label=cond)
    ctrl = results[(results["table"] == "accuracy") & results["gold_index"].isna()]
    for _, r in ctrl.iterrows():
        ax.axhline(r["estimate"], linestyle="--", linewidth=0.9, color="gray")
        ax.text(0.99, r["estimate"], r["condition"],
                transform=ax.get_yaxis_transform(), ha="right", va="bottom", fontsize=8, color="gray")
    ax.set_xticks(sorted(acc["gold_index"].unique()))
    ax.set_xlabel("Gold passage index (0 = first)")
    ax.set_ylabel("Accuracy (higher is better)")
    ax.set_title(title, fontsize=10)
    ax.legend(title="Ordering", fontsize=8, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=len(groups))
    ax.grid(alpha=0.25)
    fig.tight_layout()
    out_stem.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_stem.with_suffix(".png"), dpi=300)
    fig.savefig(out_stem.with_suffix(".pdf"))
    plt.close(fig)


def plot_flips(results: pd.DataFrame, out_stem: Path):
    """Descriptive flip rates (share of questions whose correctness changes; lower = more stable).
    No intervals are drawn; the paired tests are in results.csv."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    pr = results[results["table"] == "paired"]
    series = [("O1", "O0", "O1 vs O0 (distractor order)"), ("O2", "O0", "O2 vs O0 (distractor order)"),
              ("R", "O0", "rerun vs O0 (noise floor)")]
    positions = sorted(pr["gold_index"].dropna().astype(int).unique())
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    w = 0.26
    for k, (cond, comp, label) in enumerate(series):
        sub = pr[(pr["condition"] == cond) & pr["comparator"].str.startswith(comp + "@")]
        sub = sub[sub["comparator"] == sub["gold_index"].apply(lambda g: f"{comp}@{int(g)}")].sort_values("gold_index")
        if sub.empty:
            continue
        x = [positions.index(int(g)) + (k - 1) * w for g in sub["gold_index"]]
        ax.bar(x, sub["flip_rate"], width=w, label=label)
    ax.set_xticks(range(len(positions)))
    ax.set_xticklabels(positions)
    ax.set_xlabel("Gold passage index (0 = first)")
    ax.set_ylabel("Flip rate (lower is more stable)")
    ax.set_title("Share of questions whose correctness changes", fontsize=10)
    ax.legend(fontsize=8, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=1)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    out_stem.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_stem.with_suffix(".png"), dpi=300)
    fig.savefig(out_stem.with_suffix(".pdf"))
    plt.close(fig)

In [ ]:
%%writefile /content/work/position-order-study/src/positionorder/runner.py
"""Question selection, document assembly, resumable job running and the predictions table (cycle 6).

Nothing here loads a model: the caller passes `generate_fn(prompt) -> (text, n_prompt_tokens,
n_new_tokens, truncated)`. That keeps the logic testable offline with a dummy generator.
"""
from __future__ import annotations

import dataclasses
import json
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

from .data import QAExample, validate_example
from .scoring import first_line, score_reply
from .seeds import derive_seed, distractor_permutation

ORDERING_SEEDS = {"O1": 101, "O2": 202}   # protocol seeds
CONTROLS = ("C1", "C2")


# ----------------------------------------------------------------- question selection
def eligible_qids(by_position: Dict[int, List[QAExample]], total_documents: int = 20):
    """Questions that are present once and clean in every position file, with the same gold passage and
    the same distractor sequence in all files (amendment C-2). Returns (sorted qids, exclusion counts)."""
    positions = sorted(by_position)
    maps: Dict[int, Dict[str, List[QAExample]]] = {}
    for p in positions:
        m: Dict[str, List[QAExample]] = {}
        for ex in by_position[p]:
            m.setdefault(ex.qid, []).append(ex)
        maps[p] = m
    all_ids = set().union(*[set(m) for m in maps.values()])
    reasons = {"duplicate_in_file": 0, "missing_in_some_file": 0, "validation_problem": 0,
               "inconsistent_across_files": 0}
    ok: List[str] = []
    for q in sorted(all_ids):
        if any(len(m.get(q, [])) > 1 for m in maps.values()):
            reasons["duplicate_in_file"] += 1
            continue
        if any(q not in maps[p] for p in positions):
            reasons["missing_in_some_file"] += 1
            continue
        exs = [maps[p][q][0] for p in positions]
        if any(validate_example(ex, total_documents, p) for ex, p in zip(exs, positions)):
            reasons["validation_problem"] += 1
            continue
        golds = {tuple(x.text.strip() for x in ex.passages if x.isgold) for ex in exs}
        seqs = {tuple(x.text.strip() for x in ex.passages if not x.isgold) for ex in exs}
        if len(golds) != 1 or len(seqs) != 1:
            reasons["inconsistent_across_files"] += 1
            continue
        ok.append(q)
    return ok, reasons


def split_questions(eligible: List[str], seed: int, n_dev: int, n_eval: int) -> Dict[str, List[str]]:
    """Disjoint dev and eval lists, in a seed-determined order (take prefixes for smaller runs)."""
    if len(eligible) < n_dev + n_eval:
        raise ValueError(f"only {len(eligible)} eligible questions; need {n_dev + n_eval}")
    ids = sorted(eligible)
    rng = np.random.Generator(np.random.PCG64(derive_seed(seed, "split")))
    chosen = [ids[i] for i in rng.permutation(len(ids))[: n_dev + n_eval]]
    return {"dev": chosen[:n_dev], "eval": chosen[n_dev:]}


# ----------------------------------------------------------------- documents and jobs
def docs_for(ex: QAExample, condition: str, gold_index: Optional[int]) -> List[dict]:
    """Document dicts for one condition. O0 and R keep the file's order; O1/O2 shuffle only the distractors
    (per-question SHA-256 seed) and put the gold passage at `gold_index`; C1 has no documents; C2 only the gold."""
    passages = [dataclasses.asdict(p) for p in ex.passages]
    gold = [p for p in passages if p["isgold"]]
    dist = [p for p in passages if not p["isgold"]]
    if len(gold) != 1:
        raise ValueError(f"question {ex.qid}: expected exactly one gold passage, found {len(gold)}")
    if condition in ("O0", "R"):
        return passages
    if condition == "C1":
        return []
    if condition == "C2":
        return gold
    if condition in ORDERING_SEEDS:
        if gold_index is None:
            raise ValueError("gold_index required")
        perm = distractor_permutation(ORDERING_SEEDS[condition], ex.qid, len(dist))
        out = [dist[i] for i in perm]
        out.insert(gold_index, gold[0])
        return out
    raise ValueError(f"unknown condition {condition}")


@dataclass(frozen=True)
class Job:
    split: str
    condition: str
    gold_index: Optional[int]
    qid: str
    run_id: int = 0

    def key(self) -> str:
        return f"{self.split}|{self.condition}|{self.gold_index}|{self.qid}|{self.run_id}"


def make_jobs(split: str, qids: Iterable[str], plan: List[Tuple[str, Optional[List[int]]]]) -> List[Job]:
    """plan: [(condition, positions or None)], jobs are grouped by condition block, then position."""
    qids = list(qids)
    jobs: List[Job] = []
    for cond, positions in plan:
        for g in (positions if positions is not None else [None]):
            jobs += [Job(split, cond, g, q) for q in qids]
    return jobs


def load_rows(path) -> List[dict]:
    p = Path(path)
    rows: List[dict] = []
    if not p.exists():
        return rows
    with open(p, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                break          # a half-written last line after a disconnect: it will be redone
    return rows


def row_key(r: dict) -> str:
    return f"{r['split']}|{r['condition']}|{r['gold_index']}|{r['qid']}|{r['run_id']}"


def run_jobs(jobs: List[Job], build_prompt: Callable[[Job], str], generate_fn, out_path,
             progress_every: int = 100, log=print) -> int:
    """Run jobs not yet in `out_path`, appending one JSON line per job. Returns the number run now."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    done = {row_key(r) for r in load_rows(out_path)}
    todo = [j for j in jobs if j.key() not in done]
    log(f"{len(jobs)} jobs, {len(done)} already done, {len(todo)} to run")
    t0 = time.time()
    n = 0
    with open(out_path, "a", encoding="utf-8") as fh:
        for j in todo:
            prompt = build_prompt(j)
            t = time.time()
            text, n_prompt, n_new, truncated = generate_fn(prompt)
            fh.write(json.dumps({"split": j.split, "condition": j.condition, "gold_index": j.gold_index,
                                 "qid": j.qid, "run_id": j.run_id, "model_answer": text,
                                 "n_prompt_tokens": n_prompt, "n_new_tokens": n_new,
                                 "truncated": int(truncated), "seconds": round(time.time() - t, 4)}) + "\n")
            fh.flush()
            n += 1
            if n % progress_every == 0:
                el = time.time() - t0
                log(f"  {n}/{len(todo)} done, {el / n:.2f} s per prompt, about {(len(todo) - n) * el / n / 60:.0f} min left")
    return n


# ----------------------------------------------------------------- predictions and analysis helpers
FAILURE_RULES = [  # fixed before any result is seen; first matching rule wins
    ("empty_first_line", lambda fl: fl == ""),
    ("no_answer_or_refusal", lambda fl: re.search(
        r"\b(no information|no answer|not (mentioned|provided|specified|stated|found|clear|say)|do not|does not|did not|don't|doesn't|cannot|can't|unknown|unable)\b",
        fl.lower()) is not None),
]


def failure_category(model_answer: str, correct: int) -> str:
    if correct:
        return "correct"
    fl = first_line(model_answer)
    for name, rule in FAILURE_RULES:
        if rule(fl):
            return name
    return "other_wrong"


def to_predictions(rows: List[dict], answers_by_qid: Dict[str, List[str]], settings: dict,
                   repo_scorer=None) -> pd.DataFrame:
    """Long table in the schema of docs/evaluation_schema.md. If `repo_scorer` (the repository's
    best_subspan_em) is given, every score is cross-checked and a mismatch raises."""
    keys = [row_key(r) for r in rows]
    if len(keys) != len(set(keys)):
        raise ValueError("duplicate job keys in the raw output file")
    out = []
    for r in rows:
        ans = answers_by_qid[r["qid"]]
        c = score_reply(r["model_answer"], ans)
        if repo_scorer is not None:
            ref = int(repo_scorer(prediction=first_line(r["model_answer"]), ground_truths=ans))
            if ref != c:
                raise ValueError(f"scorer disagreement for {row_key(r)}: local {c}, repository {ref}")
        out.append({"qid": r["qid"], "split": r["split"], "condition": r["condition"],
                    "gold_index": np.nan if r["gold_index"] is None else r["gold_index"],
                    "run_id": r["run_id"], "correct": c, "n_prompt_tokens": r["n_prompt_tokens"],
                    "truncated": r["truncated"], **settings,
                    "failure": failure_category(r["model_answer"], c)})
    return pd.DataFrame(out)


def standard_pairs(positions=(0, 4, 9, 14, 19), position_pairs=((0, 9), (0, 19), (9, 19))):
    """Paired comparisons fixed in advance: order effect, position effect, rerun noise."""
    pairs = []
    for g in positions:
        pairs += [(("O1", g), ("O0", g)), (("O2", g), ("O0", g)), (("O2", g), ("O1", g)), (("R", g), ("O0", g))]
    for o in ("O0", "O1", "O2"):
        pairs += [((o, a), (o, b)) for a, b in position_pairs]
    return pairs


def filter_pairs(df: pd.DataFrame, pairs, eval_split: str = "eval"):
    have = set(map(tuple, df[df["split"] == eval_split].fillna({"gold_index": -1})[["condition", "gold_index"]]
                   .astype({"gold_index": int}).drop_duplicates().itertuples(index=False, name=None)))
    return [p for p in pairs if p[0] in have and p[1] in have]


def make_build_prompt(by_position: Dict[int, List[QAExample]], qa_fn, closedbook_fn, doc_cls):
    """Return build(job) -> prompt string, using the given prompt functions (repository or local copy)."""
    maps = {g: {e.qid: e for e in exs} for g, exs in by_position.items()}
    first = sorted(by_position)[0]

    def build(job: Job) -> str:
        ex = maps[job.gold_index if job.gold_index is not None else first][job.qid]
        if job.condition == "C1":
            return closedbook_fn(ex.question)
        docs = docs_for(ex, job.condition, job.gold_index)
        return qa_fn(ex.question, [doc_cls.from_dict(d) for d in docs],
                     mention_random_ordering=False, query_aware_contextualization=False)
    return build

In [ ]:
%%writefile /content/work/position-order-study/src/positionorder/pipeline.py
"""One full pass: development gate, evaluation runs, audit, results, figures (cycle 6).

The caller supplies a prompt builder and a generator, so this module is testable with a dummy generator.
It writes: raw_outputs.jsonl (resumable), predictions.csv, audit.json, results.csv,
fig_accuracy_by_position.png/.pdf, failure_counts.csv, pipeline_summary.json.
"""
from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List

import pandas as pd

from . import evaluate as ev
from . import runner as rn
from .scoring import score_reply


def _acc(rows, answers, split, cond, gold=None):
    sel = [r for r in rows if r["split"] == split and r["condition"] == cond and r["gold_index"] == gold]
    if not sel:
        return None, 0
    return sum(score_reply(r["model_answer"], answers[r["qid"]]) for r in sel) / len(sel), len(sel)


def run_pipeline(tag: str, out_dir, by_position, split: Dict[str, List[str]], n_dev: int, n_eval: int,
                 positions, build_prompt, generate_fn, settings: dict, repo_scorer=None, gate: bool = True,
                 n_rerun: int = 100, n_boot: int = 10000, boot_seed: int = 777, log=print,
                 orderings=("O1", "O2")) -> dict:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    raw = out / "raw_outputs.jsonl"
    first = sorted(by_position)[0]
    answers = {e.qid: e.answers for e in by_position[first]}
    dev, evl = split["dev"][:n_dev], split["eval"][:n_eval]
    positions = list(positions)
    if not {0, 9} <= set(positions):
        raise ValueError("positions must include 0 and 9 (used by the development gate)")
    summary: Dict[str, object] = {"tag": tag, "n_dev": len(dev), "n_eval": len(evl), "positions": positions,
                                  "orderings": list(orderings)}

    # ---- stage 1: development questions (debugging only; never pooled with evaluation)
    dev_jobs = rn.make_jobs("dev", dev, [("C1", None), ("C2", None), ("O0", [0, 9]), ("R", [0])])
    rn.run_jobs(dev_jobs, build_prompt, generate_fn, raw, log=log)
    rows = rn.load_rows(raw)
    c1, n1 = _acc(rows, answers, "dev", "C1")
    c2, n2 = _acc(rows, answers, "dev", "C2")
    o0a, _ = _acc(rows, answers, "dev", "O0", 0)
    o0b, _ = _acc(rows, answers, "dev", "O0", 9)
    dev_o0 = {r["qid"]: score_reply(r["model_answer"], answers[r["qid"]]) for r in rows
              if r["split"] == "dev" and r["condition"] == "O0" and r["gold_index"] == 0}
    dev_r = {r["qid"]: score_reply(r["model_answer"], answers[r["qid"]]) for r in rows
             if r["split"] == "dev" and r["condition"] == "R"}
    common = set(dev_o0) & set(dev_r)
    rerun_flips = sum(dev_o0[q] != dev_r[q] for q in common)
    same_text = sum(1 for a in rows if a["split"] == "dev" and a["condition"] == "R" for b in rows
                    if b["split"] == "dev" and b["condition"] == "O0" and b["gold_index"] == 0
                    and b["qid"] == a["qid"] and b["model_answer"] == a["model_answer"])
    summary["dev"] = {"closed_book_acc": c1, "oracle_acc": c2, "o0_pos0_acc": o0a, "o0_pos9_acc": o0b,
                      "rerun_pairs": len(common), "rerun_correctness_flips": rerun_flips,
                      "rerun_identical_outputs": same_text,
                      "mean_seconds_per_prompt": (sum(r["seconds"] for r in rows) / len(rows)) if rows else None,
                      "truncated_prompts": sum(r["truncated"] for r in rows)}
    log("DEV:", json.dumps(summary["dev"]))
    if gate:
        if c1 is None or c2 is None or not c2 > c1:
            raise RuntimeError(f"GATE FAILED: oracle accuracy ({c2}) is not above closed-book accuracy ({c1}) "
                               "on the development set. Stop and diagnose prompt, scorer or model before continuing.")
        if summary["dev"]["truncated_prompts"]:
            raise RuntimeError("GATE FAILED: truncated prompts on the development set.")

    # ---- stage 2: evaluation questions (controls, then O0, rerun, O1, O2)
    plan = [("C1", None), ("C2", None), ("O0", positions)]
    jobs = rn.make_jobs("eval", evl, plan)
    jobs += rn.make_jobs("eval", evl[:min(n_rerun, len(evl))], [("R", positions)])
    jobs += rn.make_jobs("eval", evl, [(o, positions) for o in orderings])
    rn.run_jobs(jobs, build_prompt, generate_fn, raw, log=log)

    # ---- stage 3: predictions, audit, results
    rows = rn.load_rows(raw)
    df = rn.to_predictions(rows, answers, settings, repo_scorer=repo_scorer)
    df.to_csv(out / "predictions.csv", index=False)
    rep = ev.audit(df, positions=tuple(positions), eval_split="eval")
    (out / "audit.json").write_text(json.dumps([f.__dict__ for f in rep.findings], indent=2))
    for f in rep.findings:
        log(f"[{f.level}] {f.code}: {f.message}")
    if not rep.ok:
        raise RuntimeError("AUDIT FAILED: see audit.json. No metrics were computed.")
    pairs = rn.filter_pairs(df, rn.standard_pairs(positions))
    res = ev.compute_results(df, "eval", n_boot, boot_seed, input_sha256=ev.sha256_file(out / "predictions.csv"),
                             pairs=pairs)
    res.to_csv(out / "results.csv", index=False)
    ev.plot_accuracy(res, out / "fig_accuracy_by_position", conditions=["O0", "O1", "O2"])
    fail = (df[df["split"] == "eval"].fillna({"gold_index": -1})
            .groupby(["condition", "gold_index", "failure"]).size().unstack(fill_value=0).reset_index())
    fail.to_csv(out / "failure_counts.csv", index=False)
    summary.update({"n_prediction_rows": len(df), "n_pairs": len(pairs), "audit_ok": True,
                    "results_csv": str(out / "results.csv")})
    (out / "pipeline_summary.json").write_text(json.dumps(summary, indent=2))
    return summary

In [ ]:
%%writefile /content/work/position-order-study/src/positionorder/hf_generate.py
"""Hugging Face generator for the Colab run (cycle 6). Imports torch and transformers lazily.

Returns the callable expected by runner.run_jobs: generate(prompt) -> (reply, n_prompt_tokens, n_new_tokens, truncated).

Design choices (declared deviations from the original scripts, see docs/deviation_log.md):
  * The repository prompt is sent as the single user message through the tokenizer's own chat template.
  * Greedy decoding with repetition_penalty set to 1.0 explicitly (the model's default generation
    config may differ; it is printed in the notebook and saved).
  * The reply is strip()-ed, as in the original Llama-2 script.
  * early_stop=True ends generation once the reply has a newline after its first non-blank character.
    The scorer reads only the first line, so the score is unchanged; the notebook checks this on
    development prompts before the main run.
  * No prompt is skipped or truncated; `truncated` is 1 only if prompt + new tokens exceed the context size.
"""
from __future__ import annotations


def make_hf_generator(model, tokenizer, max_new_tokens: int = 100, max_positions: int = 32768,
                      early_stop: bool = True):
    import torch
    from transformers import StoppingCriteria, StoppingCriteriaList

    class FirstLineDone(StoppingCriteria):
        def __init__(self, prompt_len: int):
            self.prompt_len = prompt_len

        def __call__(self, input_ids, scores, **kwargs):
            text = tokenizer.decode(input_ids[0, self.prompt_len:], skip_special_tokens=True)
            done = "\n" in text.lstrip()
            return torch.full((input_ids.shape[0],), done, dtype=torch.bool, device=input_ids.device)

    def generate(prompt: str):
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        n_prompt = int(inputs["input_ids"].shape[1])
        kwargs = dict(max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.0,
                      temperature=None, top_p=None, top_k=None, pad_token_id=tokenizer.eos_token_id)
        if early_stop:
            kwargs["stopping_criteria"] = StoppingCriteriaList([FirstLineDone(n_prompt)])
        with torch.inference_mode():
            out = model.generate(**inputs, **kwargs)
        new = out[0, n_prompt:]
        reply = tokenizer.decode(new, skip_special_tokens=True).strip()
        return reply, n_prompt, int(new.shape[0]), int(n_prompt + max_new_tokens > max_positions)

    return generate

In [ ]:
%%writefile /content/work/position-order-study/src/positionorder/derive.py
"""Derive the position files from the single gold-at-0 file (cycle 6).

The original script (make_qa_data_from_retrieval_results.py, read in full) builds every position
file the same way: take the first N-1 retrieved passages that do not contain the answer, in
retrieval order, and insert the gold passage at the requested index. So the file with the gold
passage at index 0 already holds, in order, every distractor of every other position file, and a
position file for index g is that file with the gold passage moved from index 0 to index g.

Two checks on the real gold-at-0 file support this reading (see docs/deviation_log.md, DEV-010).
This function does not touch the gold passage or any distractor; it only moves the gold passage.
"""
from __future__ import annotations

import gzip
import json
from pathlib import Path


class DeriveError(ValueError):
    pass


def derive_position_file(src_path, gold_index: int, out_path, total_documents: int = 20) -> int:
    """Write a gzipped JSONL file with the gold passage at `gold_index`. Returns the number of examples."""
    src_path, out_path = Path(src_path), Path(out_path)
    if not 0 <= gold_index < total_documents:
        raise DeriveError(f"gold_index {gold_index} outside 0..{total_documents - 1}")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    opener = gzip.open if str(src_path).endswith(".gz") else open
    n = 0
    with opener(src_path, "rt", encoding="utf-8") as fin, gzip.open(out_path, "wt", encoding="utf-8") as fout:
        for line_no, line in enumerate(fin, start=1):
            if not line.strip():
                continue
            ex = json.loads(line)
            ctxs = ex["ctxs"]
            if len(ctxs) != total_documents:
                raise DeriveError(f"line {line_no}: {len(ctxs)} passages, expected {total_documents}")
            golds = [i for i, c in enumerate(ctxs) if c.get("isgold") is True]
            if golds != [0]:
                raise DeriveError(f"line {line_no}: source must have exactly one gold passage, at index 0 (found {golds})")
            gold, distractors = ctxs[0], ctxs[1:]
            ex["ctxs"] = distractors[:gold_index] + [gold] + distractors[gold_index:]
            fout.write(json.dumps(ex) + "\n")
            n += 1
    return n

## 3. Original repository at a fixed commit

In [ ]:
REPO_DIR = f"{WORK}/lost-in-the-middle"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", REPO_COMMIT], check=True)
head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
assert head == REPO_COMMIT, head
sys.path.insert(0, f"{REPO_DIR}/src")
sys.path.insert(0, f"{WORK}/position-order-study/src")
import regex, pydantic   # the repository's metric module needs `regex`
from lost_in_the_middle.prompting import Document, get_qa_prompt, get_closedbook_qa_prompt
from lost_in_the_middle.metrics import best_subspan_em
print("repository commit:", head)

## 4. Data: load, hash and validate the five position files

In [ ]:
QA_DIR = QA_DIR_ON_DRIVE or f"{REPO_DIR}/qa_data"
print("contents of the data folder", QA_DIR, ":")
if os.path.isdir(QA_DIR):
    for n in sorted(os.listdir(QA_DIR)):
        print("  ", n, os.path.getsize(f"{QA_DIR}/{n}"), "bytes")
else:
    print("   (folder not found)")
def qa_path(g):
    exact = f"{QA_DIR}/nq-open-{TOTAL_DOCS}_total_documents_gold_at_{g}.jsonl.gz"
    if os.path.exists(exact) or not os.path.isdir(QA_DIR):
        return exact
    for n in sorted(os.listdir(QA_DIR)):          # tolerate a slightly different file name
        if f"{TOTAL_DOCS}_total" in n and f"gold_at_{g}." in n:
            return f"{QA_DIR}/{n}"
    return exact
def is_pointer(p):                                  # a Git LFS placeholder is a tiny text file
    with open(p, "rb") as fh:
        return fh.read(40).startswith(b"version https://git-lfs")
def usable(p): return os.path.exists(p) and os.path.getsize(p) > 10_000 and not is_pointer(p)
pointers = [g for g in ALL_POSITIONS if os.path.exists(qa_path(g)) and is_pointer(qa_path(g))]
if pointers:
    print("these files are Git LFS placeholders, fetching the real files from GitHub:", pointers)
    subprocess.run("apt-get -qq install -y git-lfs", shell=True, capture_output=True)
    subprocess.run(["git", "-C", REPO_DIR, "lfs", "install", "--local"], capture_output=True, text=True)
    r = subprocess.run(["git", "-C", REPO_DIR, "lfs", "pull", "--include", "qa_data/*"], capture_output=True, text=True)
    print(r.stdout[-300:], r.stderr[-300:])
def fetch_from_drive_link(link, dest):
    """Download one Drive file by its share link: first with gdown (public links), then with your own Google sign-in."""
    import re
    m = re.search(r"/d/([A-Za-z0-9_-]+)", link) or re.search(r"[?&]id=([A-Za-z0-9_-]+)", link)
    if not m:
        raise ValueError("no file id found in the Drive link")
    fid = m.group(1)
    big = lambda: os.path.exists(dest) and os.path.getsize(dest) > 1_000_000
    if not big():
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        try:
            import gdown
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
            import gdown
        try:
            gdown.download(id=fid, output=dest, quiet=False)
        except Exception as exc:
            print("gdown could not download the file:", exc)
    if not big():
        print("trying the Google Drive API with your own sign-in (a permission window may open)")
        from google.colab import auth
        auth.authenticate_user()
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        with open(dest, "wb") as fh:
            dl = MediaIoBaseDownload(fh, build("drive", "v3").files().get_media(fileId=fid))
            done = False
            while not done:
                _, done = dl.next_chunk()
    if not big():
        raise RuntimeError("Could not download the file from the Drive link. Open the link, choose Add shortcut to Drive or Make a copy, "
                           "then set GOLD0_FILE_ON_DRIVE to the path of that copy and Run all again.")
    with open(dest, "rb") as fh:
        if fh.read(2) != b"\x1f\x8b":
            raise RuntimeError("The downloaded file is not a gzip file (a Google sign-in or permission page was probably saved). "
                               "Make the link accessible to you, or copy the file into your Drive and use GOLD0_FILE_ON_DRIVE.")
    return dest
if GOLD0_DRIVE_LINK and not GOLD0_FILE_ON_DRIVE:
    GOLD0_FILE_ON_DRIVE = fetch_from_drive_link(GOLD0_DRIVE_LINK, f"{WORK}/gold0/nq-open-{TOTAL_DOCS}_total_documents_gold_at_0.jsonl.gz")
missing = [g for g in ALL_POSITIONS if not usable(qa_path(g))]
src0 = GOLD0_FILE_ON_DRIVE or (qa_path(0) if usable(qa_path(0)) else "")
if src0 and (GOLD0_FILE_ON_DRIVE or missing):
    from positionorder.derive import derive_position_file
    QA_DIR = f"{WORK}/derived_qa"
    for g in ALL_POSITIONS:
        n_ex = derive_position_file(src0, g, qa_path(g), TOTAL_DOCS)
    with open(src0, "rb") as fh:
        SRC0_SHA = hashlib.sha256(fh.read()).hexdigest()
    if GOLD0_EXPECTED_SHA256 and SRC0_SHA != GOLD0_EXPECTED_SHA256:
        print("NOTE: this gold-at-0 file differs from the copy inspected when the notebook was built (hash mismatch). "
              "It may be a different version; compare the printed validation counts with the expected 2,655 questions.")
    print("derived the five position files from the gold-at-0 file", src0, "| sha256", SRC0_SHA, "|", n_ex, "questions")
    missing = []
else:
    SRC0_SHA = ""
print("position files found in the repository:", [g for g in ALL_POSITIONS if g not in missing], "| missing:", missing)
if missing:   # fallback: build them with the repository's own script (needs the large retrieval file)
    RET = RETRIEVAL_FILE_ON_DRIVE or f"{WORK}/nq-open-contriever-msmarco-retrieved-documents.jsonl.gz"
    URL = "https://nlp.stanford.edu/data/nfliu/lost-in-the-middle/nq-open-contriever-msmarco-retrieved-documents.jsonl.gz"
    if not os.path.exists(RET):
        got = False
        for attempt in range(3):
            r = subprocess.run(["curl", "-sS", "-L", "--fail", "--retry", "3", "--max-time", "3600", "-o", RET, URL],
                               capture_output=True, text=True)
            if r.returncode == 0 and os.path.exists(RET) and os.path.getsize(RET) > 1_000_000:
                got = True
                break
            print("download attempt", attempt + 1, "failed: exit", r.returncode, r.stderr[:300])
        if not got:
            raise RuntimeError("Could not download the retrieval file. Download it in your own browser from the README link, "
                               "upload it to Google Drive, set RETRIEVAL_FILE_ON_DRIVE in the settings cell to its path, and Run all again.")
    for g in missing:
        subprocess.run([sys.executable, "-u", f"{REPO_DIR}/scripts/make_qa_data_from_retrieval_results.py",
                        "--input-path", RET, "--num-total-documents", str(TOTAL_DOCS),
                        "--gold-index", str(g), "--output-path", qa_path(g)], check=True)
data_hashes = {"source_gold0_sha256": SRC0_SHA}
for g in ALL_POSITIONS:
    with open(qa_path(g), "rb") as fh:
        h = hashlib.sha256(fh.read()).hexdigest()
    data_hashes[g] = {"file": os.path.basename(qa_path(g)), "bytes": os.path.getsize(qa_path(g)), "sha256": h}
    print(g, data_hashes[g])

In [ ]:
from positionorder.data import load_qa_file, validate_file, check_across_positions
by_position, file_reports = {}, []
for g in ALL_POSITIONS:
    exs = load_qa_file(qa_path(g))
    by_position[g] = exs
    rep = validate_file(qa_path(g), TOTAL_DOCS, expected_gold_index=g, examples=exs)
    file_reports.append(rep)
    probs = {k: v for k, v in rep["problem_counts"].items() if v}
    print(f"gold_at_{g}: {rep['n_examples']} examples, {rep['n_unique_questions']} unique questions, {rep['n_clean']} clean, problems: {probs}")
cross = check_across_positions(by_position)
print({k: cross[k] for k in ("n_questions_in_all_files", "n_questions_missing_from_some_file",
                             "n_gold_text_differs", "n_distractor_set_differs", "n_distractor_sequence_differs")})
json.dump({"files": file_reports, "cross_position": cross, "data": data_hashes},
          open(f"{OUT}/data_validation_report.json", "w"), indent=2)

## 5. Eligible questions and the seeded development / evaluation split

In [ ]:
from positionorder import runner as rn
from positionorder.scoring import normalize_answer, risky_answers, score_reply, first_line, cross_check
eligible, reasons = rn.eligible_qids(by_position, TOTAL_DOCS)
answers = {e.qid: e.answers for e in by_position[ALL_POSITIONS[0]]}
questions = {e.qid: e.question for e in by_position[ALL_POSITIONS[0]]}
empty_norm = {q for q in eligible if any(normalize_answer(a) == "" for a in answers[q])}
eligible = [q for q in eligible if q not in empty_norm]
print("eligible questions:", len(eligible), "| exclusions:", reasons,
      "| excluded because an accepted answer is empty after normalisation:", len(empty_norm))
split = rn.split_questions(eligible, SPLIT_SEED, 100, 300)    # always the full-size split; smaller modes use prefixes
assert not set(split["dev"]) & set(split["eval"]), "dev and eval overlap"
risky = {k: sum(1 for q in v if risky_answers(answers[q])) for k, v in split.items()}
print("questions with a very short accepted answer (<4 characters after normalisation):", risky)
json.dump(split, open(f"{OUT}/split.json", "w"))

## 6. Prompt and scorer checks against the original code

In [ ]:
from positionorder import prompts_local as pl
build = rn.make_build_prompt(by_position, get_qa_prompt, get_closedbook_qa_prompt, Document)
build_local = rn.make_build_prompt(by_position, pl.get_qa_prompt, pl.get_closedbook_qa_prompt, pl.Document)
test_jobs = rn.make_jobs("dev", split["dev"][:20], [("C1", None), ("C2", None), ("O0", POSITIONS), ("O1", POSITIONS), ("O2", POSITIONS)])
bad = [j for j in test_jobs if build(j) != build_local(j)]
assert not bad, f"{len(bad)} prompts differ between the repository functions and the local copy"
maps = {g: {e.qid: e for e in by_position[g]} for g in POSITIONS}
for q in split["dev"][:20]:
    for g in POSITIONS:
        d0 = rn.docs_for(maps[g][q], "O0", g)
        assert d0[g]["isgold"]
        for c in ("O1", "O2"):
            d = rn.docs_for(maps[g][q], c, g)
            assert d[g]["isgold"] and len(d) == TOTAL_DOCS
            assert sorted(x["text"] for x in d) == sorted(x["text"] for x in d0)
print("prompt builder OK on", len(test_jobs), "jobs. Example oracle (C2) prompt:\n")
print(build(rn.make_jobs("dev", [split["dev"][0]], [("C2", None)])[0])[:700])
vectors = [("Paris\nLondon", ["Paris"]), ("\nParis", ["Paris"]), ("The Eiffel-Tower!", ["eiffel tower"]),
           ("start", ["art"]), ("anything", ["the"]), ("Barack Obama.", ["Obama", "Barack"]),
           ("It was in 1990.", ["1990"]), ("none", ["x"])]
assert cross_check(vectors, best_subspan_em) == [], "local scorer and the repository metric disagree"
print("scorer agrees with the repository metric on", len(vectors), "test replies")

## 7. Freeze record (written before any model output exists)

In [ ]:
import glob
code_hashes = {}
for p in sorted(glob.glob(f"{WORK}/position-order-study/src/positionorder/*.py")):
    with open(p, "rb") as fh:
        code_hashes[os.path.basename(p)] = hashlib.sha256(fh.read()).hexdigest()
with open(f"{OUT}/split.json", "rb") as fh:
    split_hash = hashlib.sha256(fh.read()).hexdigest()
freeze = {"written_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "tag": TAG, "model_id": MODEL_ID,
          "repo_commit": REPO_COMMIT, "mode": MODE, "positions": POSITIONS, "orderings": ["O0"] + ORDERINGS, "seeds": {"split": SPLIT_SEED, "bootstrap": BOOT_SEED,
          "O1": 101, "O2": 202}, "n_dev": N_DEV, "n_eval": N_EVAL, "n_rerun": N_RERUN, "max_new_tokens": MAX_NEW_TOKENS,
          "template": "qa.prompt for every condition; closedbook_qa.prompt for C1", "split_sha256": split_hash,
          "code_sha256": code_hashes, "data": data_hashes,
          "decision_rule_PROPOSED_author_must_confirm_before_reading_results":
              "an effect is called meaningful only if its 95% bootstrap interval excludes zero, it exceeds 5 percentage points, "
              "and the corresponding order or position flip rate exceeds the rerun flip rate"}
fp = f"{OUT}/freeze.json"
if os.path.exists(fp):
    old = json.load(open(fp))
    changed = [k for k in code_hashes if old.get("code_sha256", {}).get(k) != code_hashes[k]]
    print("freeze.json already exists (kept). Code files that differ from it:", changed or "none")
    if old.get("split_sha256") != split_hash:
        print("WARNING: the split differs from the frozen split!")
else:
    json.dump(freeze, open(fp, "w"), indent=2)
    print("freeze.json written:", fp)

## 8. Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_ID)
dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map="cuda")
model.eval()
try:
    from huggingface_hub import model_info
    REV = model_info(MODEL_ID).sha
except Exception as exc:
    REV = "unknown"
    print("could not read the model revision:", exc)
MAX_POS = int(model.config.max_position_embeddings)
print("model", MODEL_ID, "| revision", REV, "| dtype", dtype, "| max positions", MAX_POS)
print("model default generation config (overridden for greedy decoding with repetition_penalty=1.0):")
print(model.generation_config)

## 9. Smoke test, early-stop equivalence, and a time estimate

In [ ]:
from positionorder.hf_generate import make_hf_generator
gen = make_hf_generator(model, tok, MAX_NEW_TOKENS, MAX_POS, early_stop=True)
gen_full = make_hf_generator(model, tok, MAX_NEW_TOKENS, MAX_POS, early_stop=False)
q0 = split["dev"][0]
for cond, g in [("C2", None), ("O0", 0), ("C1", None)]:
    j = rn.make_jobs("dev", [q0], [(cond, [g] if g is not None else None)])[0]
    t = time.time()
    reply, n_p, n_new, trunc = gen(build(j))
    print(f"{cond} {g} | prompt tokens {n_p} | new tokens {n_new} | {time.time() - t:.2f} s | reply {reply!r} | accepted {answers[q0]}")
    if cond == "C2":
        assert reply.strip(), "empty reply on the oracle prompt: check the dtype and chat template"

In [ ]:
# Early stopping at the first newline must not change any score (the scorer reads only the first line).
n_eq = 20 if MODE == "quick" else 30
mism = []
for q in split["dev"][:n_eq]:
    p = build(rn.make_jobs("dev", [q], [("O0", [9])])[0])
    a, b = gen(p)[0], gen_full(p)[0]
    if score_reply(a, answers[q]) != score_reply(b, answers[q]) or first_line(a) != first_line(b):
        mism.append(q)
assert not mism, f"early stopping changed the first line or score for {len(mism)} of {n_eq} prompts: set early_stop=False in make_hf_generator"
print("early-stop equivalence OK on", n_eq, "prompts")
t = time.time()
for q in split["dev"][:5]:
    gen(build(rn.make_jobs("dev", [q], [("O0", [9])])[0]))
sec = (time.time() - t) / 5
print(f"about {sec:.2f} s per prompt (5 prompts); {N_PROMPTS} prompts in this mode -> roughly {sec * N_PROMPTS / 3600:.1f} hours if the speed holds (early prompts may be faster than average)")

## 10. Main run (resumable) and audit

In [ ]:
from positionorder import pipeline as pp
settings = dict(model_id=f"{MODEL_ID}@{REV}", template_id="qa.prompt+tokenizer_chat_template",
                scorer_version=f"repo@{REPO_COMMIT[:8]} best_subspan_em first_line", decoding="greedy repetition_penalty=1.0",
                max_new_tokens=MAX_NEW_TOKENS, ctx_size=MAX_POS)
summary = pp.run_pipeline(TAG, OUT, by_position, split, N_DEV, N_EVAL, POSITIONS, build, gen, settings,
                          repo_scorer=best_subspan_em, n_rerun=N_RERUN, n_boot=10000, boot_seed=BOOT_SEED,
                          orderings=tuple(ORDERINGS))
print(json.dumps(summary, indent=2))

## 11. Extra outputs: flip-rate figure, manual audit sample, environment record, bundle

In [ ]:
import pandas as pd, numpy as np, shutil
from pathlib import Path
from positionorder import evaluate as ev
from positionorder.seeds import derive_seed
res = pd.read_csv(f"{OUT}/results.csv")
ev.plot_flips(res, Path(OUT) / "fig_flip_rates")
rows = rn.load_rows(f"{OUT}/raw_outputs.jsonl")
sample = []
for cond in ("O0", "O1", "O2"):
    for label in (1, 0):
        pool = [r for r in rows if r["split"] == "eval" and r["condition"] == cond
                and score_reply(r["model_answer"], answers[r["qid"]]) == label]
        order = np.random.Generator(np.random.PCG64(derive_seed(BOOT_SEED, "audit", cond, label))).permutation(len(pool))[:50]
        for i in order:
            r = pool[int(i)]
            sample.append({"qid": r["qid"], "condition": cond, "gold_index": r["gold_index"], "question": questions[r["qid"]],
                           "first_line": first_line(r["model_answer"]), "accepted_answers": " | ".join(answers[r["qid"]]),
                           "scored_correct": label, "manual_label": ""})
pd.DataFrame(sample).to_csv(f"{OUT}/manual_audit_sample.csv", index=False)
try:
    driver_info = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,memory.total", "--format=csv"],
                                 capture_output=True, text=True).stdout
except Exception as exc:
    driver_info = f"unavailable: {exc}"
env = {"python": sys.version, "torch": torch.__version__, "transformers": transformers.__version__, "pydantic": pydantic.__version__,
       "gpu": torch.cuda.get_device_name(0), "dtype": str(dtype), "model_id": MODEL_ID, "model_revision": REV,
       "max_positions": MAX_POS, "repo_commit": REPO_COMMIT, "generation_config": str(model.generation_config),
       "driver_info": driver_info}
json.dump(env, open(f"{OUT}/environment.json", "w"), indent=2)
with open(f"{OUT}/pip_freeze.txt", "w") as fh:
    fh.write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
bundle = shutil.make_archive(f"{DRIVE_DIR}/results_bundle_{TAG}", "zip", OUT)
print("files in", OUT, ":", sorted(os.listdir(OUT)))
print("bundle:", bundle)
try:
    from google.colab import files
    files.download(bundle)
except Exception as exc:
    print("automatic download not available; take the bundle from Google Drive:", exc)

## What to do with the output
Send the bundle (or at least `results.csv`, `predictions.csv`, `audit.json`, `pipeline_summary.json`, `freeze.json`, `environment.json`, `failure_counts.csv`, the figures and the printed output of the cells above). Do not read any number as a finding until the audit passed, the development gate passed, and the decision rule in `freeze.json` was confirmed by the author *before* looking at results. After `manual_audit_sample.csv` is labelled by hand, the scorer error rate can be reported.